In [5]:
import pandas as pd
from sklearn.utils import resample

# 1. Load the cleaned data
file_path = r"C:\Users\Tejas\Desktop\hybrid_recommender\data\processed\cleaned_reviews.csv"
print("Loading data...")
df = pd.read_csv(file_path)

# Drop rows where the text might be completely missing after cleaning
df = df.dropna(subset=['clean_text'])

# 2. Engineer the Target Labels
# We drop 3-star reviews because "It was okay" doesn't help us learn strong sentiment.
df = df[df['Score'] != 3].copy()

# Create a new column: 1 for Positive (4, 5), 0 for Negative (1, 2)
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

# 3. Check the massive imbalance
positive_reviews = df[df['Sentiment'] == 1]
negative_reviews = df[df['Sentiment'] == 0]
print(f"BEFORE balancing:")
print(f"Positive Reviews: {len(positive_reviews)}")
print(f"Negative Reviews: {len(negative_reviews)}")

# 4. Handle Imbalance via Undersampling
print("\nUndersampling the majority class to prevent model bias...")
positive_downsampled = resample(positive_reviews, 
                                replace=False, # Don't sample the same review twice
                                n_samples=len(negative_reviews), # Match the smaller negative count
                                random_state=42) # Keeps it reproducible

# 5. Combine and shuffle the new perfectly balanced dataset
balanced_df = pd.concat([positive_downsampled, negative_reviews])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAFTER balancing:")
print(f"Positive Reviews: {len(balanced_df[balanced_df['Sentiment'] == 1])}")
print(f"Negative Reviews: {len(balanced_df[balanced_df['Sentiment'] == 0])}")

Loading data...
BEFORE balancing:
Positive Reviews: 443777
Negative Reviews: 82037

Undersampling the majority class to prevent model bias...

AFTER balancing:
Positive Reviews: 82037
Negative Reviews: 82037


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np

# 1. Engineering Constants
MAX_LEN = 160
MAX_WORDS = 20000

print("Building the Vocabulary (this might take a few seconds)...")

# 2. Count every single word in our balanced dataset
all_words = ' '.join(balanced_df['clean_text'].astype(str)).split()
word_counts = Counter(all_words)

# 3. Create the integer mapping (Vocabulary)
# We start assigning numbers at 2. 
# 0 is reserved for [PADDING], 1 is reserved for [UNKNOWN] words.
vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common(MAX_WORDS))}
vocab_size = len(vocab) + 2

print(f"Vocabulary built! Total unique words kept: {len(vocab)}")

# 4. Define the Tokenizer function
def text_to_tensor(text):
    # Convert text to a list of numbers (use 1 if the word isn't in our top 20k)
    sequence = [vocab.get(word, 1) for word in str(text).split()]
    
    # The Conveyer Belt: Truncate to 160, or Pad with 0s to 160
    if len(sequence) >= MAX_LEN:
        sequence = sequence[:MAX_LEN]
    else:
        sequence = sequence + [0] * (MAX_LEN - len(sequence))
        
    return sequence

print("Converting 164,000 reviews into mathematical sequences...")
# Process the entire dataset into lists of integers
sequences = np.array([text_to_tensor(text) for text in balanced_df['clean_text']])
labels = balanced_df['Sentiment'].values

# 5. Split into Training (80%) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(sequences, labels, test_size=0.2, random_state=42)

# 6. Create the PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        # Labels must be float32 for Binary Cross Entropy loss later
        self.labels = torch.tensor(labels, dtype=torch.float32)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# Create DataLoaders
# A batch size of 256 is a safe and fast sweet-spot for a GTX 1650 processing text
train_loader_dl = DataLoader(SentimentDataset(X_train, y_train), batch_size=256, shuffle=True)
val_loader_dl = DataLoader(SentimentDataset(X_val, y_val), batch_size=256, shuffle=False)

print("Deep Learning DataLoaders are ready!")

c:\Users\Tejas\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Building the Vocabulary (this might take a few seconds)...
Vocabulary built! Total unique words kept: 20000
Converting 164,000 reviews into mathematical sequences...
Deep Learning DataLoaders are ready!


In [7]:
import torch.nn as nn
import torch.nn.functional as F

class SentimentCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_filters=128, kernel_size=5):
        super(SentimentCNN, self).__init__()
        
        # 1. The Lookup Table (Translates integers to 64-number vectors)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 2. The Scanner (Looks at 5 words at a time)
        # in_channels is our embed_dim. out_channels is how many "patterns" it can memorize.
        self.conv1d = nn.Conv1d(in_channels=embed_dim, 
                                out_channels=num_filters, 
                                kernel_size=kernel_size)
        
        # 3. The Final Decision Maker
        # Takes the strongest signals from the 128 filters and squashes them into 1 final score
        self.fc = nn.Linear(num_filters, 1)

    def forward(self, x):
        # x shape: (Batch_Size, 160 words)
        
        # Step A: Embed the text
        embedded = self.embedding(x) 
        # embedded shape: (Batch_Size, 160, 64)
        
        # Step B: PyTorch Conv1d expects the layout (Batch, Channels, Length)
        # So we swap the last two dimensions to match what the CNN expects
        embedded = embedded.permute(0, 2, 1) 
        
        # Step C: Slide the CNN filter and apply ReLU (turn off negative numbers)
        conved = F.relu(self.conv1d(embedded))
        
        # Step D: Global Max Pooling - grab the single most important feature from the whole text
        pooled = F.max_pool1d(conved, kernel_size=conved.shape[2]).squeeze(2)
        
        # Step E: Make the final positive/negative prediction
        return self.fc(pooled).squeeze(1)

# Instantiate the model
# vocab_size was calculated in Cell 2 (20002: 20k words + padding + unknown)
cnn_model = SentimentCNN(vocab_size=vocab_size)
print("1D CNN Architecture Built!")
print(cnn_model)

1D CNN Architecture Built!
SentimentCNN(
  (embedding): Embedding(20002, 64)
  (conv1d): Conv1d(64, 128, kernel_size=(5,), stride=(1,))
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [8]:
import torch.optim as optim

# 1. Hardware Optimization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")
cnn_model = cnn_model.to(device)

# 2. Binary Classification Loss Function and Optimizer
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

epochs = 5
train_losses, val_losses = [], []
val_accuracies = []

def calculate_accuracy(predictions, labels):
    # BCEWithLogits outputs raw numbers. We apply sigmoid to force them between 0 and 1.
    # Anything > 0.5 is predicted as Positive (1), else Negative (0).
    rounded_preds = torch.round(torch.sigmoid(predictions))
    correct = (rounded_preds == labels).float() 
    acc = correct.sum() / len(correct)
    return acc

print("Starting CNN Training Loop...")

for epoch in range(epochs):
    # --- TRAINING ---
    cnn_model.train()
    total_train_loss = 0
    
    for sequences, labels in train_loader_dl:
        sequences, labels = sequences.to(device), labels.to(device)
        
        optimizer.zero_grad()
        predictions = cnn_model(sequences)
        loss = criterion(predictions, labels)
        
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader_dl)
    train_losses.append(avg_train_loss)
    
    # --- VALIDATION ---
    cnn_model.eval()
    total_val_loss = 0
    total_val_acc = 0
    
    with torch.no_grad():
        for sequences, labels in val_loader_dl:
            sequences, labels = sequences.to(device), labels.to(device)
            
            predictions = cnn_model(sequences)
            loss = criterion(predictions, labels)
            acc = calculate_accuracy(predictions, labels)
            
            total_val_loss += loss.item()
            total_val_acc += acc.item()
            
    avg_val_loss = total_val_loss / len(val_loader_dl)
    avg_val_acc = total_val_acc / len(val_loader_dl)
    
    val_losses.append(avg_val_loss)
    val_accuracies.append(avg_val_acc)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {avg_val_acc*100:.2f}%")

print("Deep Learning Training Complete!")

Training on device: cuda
Starting CNN Training Loop...
Epoch 1/5 | Train Loss: 0.3940 | Val Loss: 0.2802 | Val Accuracy: 88.48%
Epoch 2/5 | Train Loss: 0.2220 | Val Loss: 0.2254 | Val Accuracy: 91.27%
Epoch 3/5 | Train Loss: 0.1591 | Val Loss: 0.2081 | Val Accuracy: 92.09%
Epoch 4/5 | Train Loss: 0.1183 | Val Loss: 0.2146 | Val Accuracy: 91.96%
Epoch 5/5 | Train Loss: 0.0866 | Val Loss: 0.2020 | Val Accuracy: 92.58%
Deep Learning Training Complete!


In [9]:
import os
import torch

# 1. Define the save path directly in your models folder
model_dir = r"C:\Users\Tejas\Desktop\hybrid_recommender\models"
save_path = os.path.join(model_dir, "sentiment_cnn.pth")

# 2. Save the PyTorch state dictionary (the learned weights)
torch.save(cnn_model.state_dict(), save_path)

print(f"Success! CNN model weights saved to: {save_path}")

Success! CNN model weights saved to: C:\Users\Tejas\Desktop\hybrid_recommender\models\sentiment_cnn.pth


In [10]:
import json
import os

model_dir = r"C:\Users\Tejas\Desktop\hybrid_recommender\models"
vocab_path = os.path.join(model_dir, "vocab.json")

# Save the exact dictionary we used to train the CNN
with open(vocab_path, 'w') as f:
    vocab = globals().get("vocab")
    if vocab is None:
        source_df = globals().get("balanced_df", globals().get("df"))
        if source_df is None:
            raise NameError("Neither 'balanced_df' nor 'df' is defined.")
        all_words = " ".join(source_df["clean_text"].astype(str)).split()
        word_counts = Counter(all_words)
        vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(MAX_WORDS))}

    json.dump(vocab, f)

print(f"Enterprise fix: Vocabulary permanently saved to {vocab_path}")

Enterprise fix: Vocabulary permanently saved to C:\Users\Tejas\Desktop\hybrid_recommender\models\vocab.json


In [11]:
import json
import os

model_dir = r"C:\Users\Tejas\Desktop\hybrid_recommender\models"
vocab_path = os.path.join(model_dir, "vocab.json")

# Save the exact dictionary we used to train the CNN
with open(vocab_path, 'w') as f:
    json.dump(vocab, f)

print(f"Enterprise fix: Vocabulary permanently saved to {vocab_path}")

Enterprise fix: Vocabulary permanently saved to C:\Users\Tejas\Desktop\hybrid_recommender\models\vocab.json
